# Is that decay time, or is it population mix?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [13]:
import os

import numpy as np
import polars as pl
from google.cloud import bigquery, storage
from sklearn.metrics import average_precision_score, roc_auc_score

from fraud_detection.core.config import resolve_repo_path
from fraud_detection.core.feature_contract.admission import load_admission_rules
from fraud_detection.core.promotion import parse_promotion_marker
from fraud_detection.core.schema import MODEL_INPUT_TABLE, SPLIT_TABLE
from fraud_detection.feature_engineering.derivations import apply_derivations
from fraud_detection.training.data import load_raw_split, prepare_features, to_lightgbm

PROJECT = os.environ["GCP_PROJECT_ID"]
bq = bigquery.Client(project=PROJECT)
gcs = storage.Client(project=PROJECT).bucket(f"{PROJECT}-models")

In [10]:
# The promoted model, read the way the scoring path reads it — the marker, never the
# newest artifact. Analysing a model nobody promoted would describe a decision nobody made.
import pickle

promoted = parse_promotion_marker(gcs.blob("promoted/production.json").download_as_text())
print(f"{promoted.run}  contract {promoted.contract_fingerprint}  code {promoted.code_version[:12]}")

bundle = pickle.loads(gcs.blob(f"lightgbm/{promoted.run}/model.pkl").download_as_bytes())
booster = bundle["booster"]

raw = load_raw_split(bq, PROJECT, "test", model_input_table=MODEL_INPUT_TABLE, split_table=SPLIT_TABLE)
derived = apply_derivations(raw, load_admission_rules().derivations)
features = prepare_features(derived)
scores = booster.predict(to_lightgbm(features.select(booster.feature_name())),
                         num_iteration=booster.best_iteration)

frame = raw.select(["TransactionID", "isFraud", "TransactionAmt", "TransactionDT",
                    "ProductCD", "card1", "addr1", "D1", "D9"]).with_columns(
    score=pl.Series(scores)
)
print(frame.shape)

ef077ec2  contract bdb97707da05adff  code 89145b690b7b
(59054, 10)


In [4]:
MIN_ROWS, MIN_POSITIVES = 500, 20


def by_segment(df: pl.DataFrame, column: str) -> pl.DataFrame:
    """PR-AUC within each level of `column`, with its own base rate beside it.

    Segments too small to estimate are reported as null rather than dropped: a segment
    nobody can measure is a finding about coverage, and silently omitting it would make
    the table look more complete than the data is.
    """
    rows = []
    for (level,), group in df.group_by([column], maintain_order=True):
        y, s = group["isFraud"].to_numpy(), group["score"].to_numpy()
        base = float(y.mean())
        measurable = len(group) >= MIN_ROWS and y.sum() >= MIN_POSITIVES
        pr = float(average_precision_score(y, s)) if measurable else None
        rows.append({
            column: level, "rows": len(group), "positives": int(y.sum()),
            "base_rate": round(base, 4),
            "pr_auc": None if pr is None else round(pr, 4),
            "lift_over_base": None if pr is None or base == 0 else round(pr / base, 2),
        })
    return pl.DataFrame(rows).sort("rows", descending=True)


overall = average_precision_score(frame["isFraud"].to_numpy(), frame["score"].to_numpy())
print(f"overall test PR-AUC {overall:.4f}, ROC-AUC "
      f"{roc_auc_score(frame['isFraud'].to_numpy(), frame['score'].to_numpy()):.4f}")

overall test PR-AUC 0.5308, ROC-AUC 0.8963


In [53]:
# Amount decile. The cost model prices a missed fraud at the transaction's full amount, so
# a model that is weak on the top decile is expensive in a way the headline metric hides.
banded = frame.with_columns(
    amt_decile=pl.col("TransactionAmt").qcut(10, labels=[f"{i}" for i in range(10)], allow_duplicates=True)
)

by_segment(banded, "amt_decile").sort("amt_decile")

amt_decile,rows,positives,base_rate,pr_auc,lift_over_base
str,i64,i64,f64,f64,f64
"""0""",5916,438,0.074,0.6422,8.67
"""1""",6301,218,0.0346,0.6469,18.7
"""2""",6701,222,0.0331,0.4869,14.7
"""3""",5368,94,0.0175,0.5074,28.98
"""4""",5248,164,0.0312,0.4549,14.56
"""5""",6803,254,0.0373,0.5413,14.5
"""6""",5001,100,0.02,0.4351,21.76
"""7""",5905,213,0.0361,0.482,13.36
"""8""",5967,185,0.031,0.4453,14.36


In [54]:
# Product, and hour of day. D9 is the hour as a fraction of one -- 24 distinct values --
# which is why it survives as a feature where the other D columns do not.
# print(by_segment(frame, "ProductCD"))
by_segment(frame, "ProductCD").sort("ProductCD")

ProductCD,rows,positives,base_rate,pr_auc,lift_over_base
str,i64,i64,f64,f64,f64
"""C""",6481,951,0.1467,0.7265,4.95
"""H""",1772,112,0.0632,0.4534,7.17
"""R""",2938,137,0.0466,0.8319,17.84
"""S""",2437,113,0.0464,0.7094,15.3
"""W""",45426,900,0.0198,0.213,10.75


In [55]:
hourly = frame.with_columns(hour=(pl.col("D9") * 24).round(0).cast(pl.Int32, strict=False))
by_segment(hourly.filter(pl.col("hour").is_not_null()), "hour").sort("hour")

hour,rows,positives,base_rate,pr_auc,lift_over_base
i64,i64,i64,f64,f64,f64
0,511,52,0.1018,0.6828,6.71
1,388,55,0.1418,null,null
2,344,56,0.1628,null,null
3,303,47,0.1551,null,null
4,218,42,0.1927,null,null
…,…,…,…,…,…
19,398,50,0.1256,null,null
20,516,71,0.1376,0.821,5.97
21,405,60,0.1481,null,null


In [56]:
# Entity age: how many days of history the card has at the moment it is scored. This is the
# axis the velocity features are built along, so it is where they should show up -- or not.
aged = frame.with_columns(
    card_age_band=pl.when(pl.col("D1").is_null()).then(pl.lit("unknown"))
    .when(pl.col("D1") < 1).then(pl.lit("0: first day"))
    .when(pl.col("D1") < 30).then(pl.lit("1: <30d"))
    .when(pl.col("D1") < 180).then(pl.lit("2: 30-180d"))
    .otherwise(pl.lit("3: 180d+"))
)
by_segment(aged, "card_age_band").sort("card_age_band")

card_age_band,rows,positives,base_rate,pr_auc,lift_over_base
str,i64,i64,f64,f64,f64
"""0: first day""",24579,1257,0.0511,0.591,11.56
"""1: <30d""",8350,520,0.0623,0.4038,6.48
"""2: 30-180d""",13563,295,0.0218,0.5683,26.13
"""3: 180d+""",12521,135,0.0108,0.5528,51.27
"""unknown""",41,6,0.1463,null,null


In [46]:
threshold_raw = float(np.quantile(frame["score"].to_numpy(), 0.974))  # ≈ the 2.6% block rate
missed = frame.filter((pl.col("isFraud") == 1) & (pl.col("score") < threshold_raw))
caught = frame.filter((pl.col("isFraud") == 1) & (pl.col("score") >= threshold_raw))

print(f"missed {len(missed):,} of {frame['isFraud'].sum():,} frauds "
      f"({len(missed) / frame['isFraud'].sum():.1%}), "
      f"carrying {missed['TransactionAmt'].sum():,.0f} of "
      f"{frame.filter(pl.col('isFraud') == 1)['TransactionAmt'].sum():,.0f} in value")


missed 1,244 of 2,213 frauds (56.2%), carrying 227,896 of 341,453 in value


In [57]:
pl.DataFrame({
    "statistic": ["median amount", "median card age (D1)", "share ProductCD=W"],
    "missed": [missed["TransactionAmt"].median(), missed["D1"].median(),
               (missed["ProductCD"] == "W").mean()],
    "caught": [caught["TransactionAmt"].median(), caught["D1"].median(),
               (caught["ProductCD"] == "W").mean()],
})


statistic,missed,caught
str,f64,f64
"""median amount""",77.0,53.18
"""median card age (D1)""",1.0,0.0
"""share ProductCD=W""",0.614148,0.140351


In [59]:
from fraud_detection.core.feature_contract import FeatureContract
from fraud_detection.evaluation.entity_purity import Anchor, EntityKey, seen_entity_flag
from fraud_detection.training.data import split_with_contract
from fraud_detection.training.model import train_lightgbm

contract = FeatureContract.from_json(resolve_repo_path("references/feature-contract.json").read_text())
rules = load_admission_rules()
full = load_raw_split(bq, PROJECT, "train", model_input_table=MODEL_INPUT_TABLE, split_table=SPLIT_TABLE)

# One training window, several validation windows at increasing gap. Same fit each time --
# only the evaluation window moves, so the curve is decay and not run-to-run noise.
dt = full["TransactionDT"]
# Anchored on the full time axis, not on the train split. Taking 0.60 of a split
# that already ends at 0.75 of the axis left no runway: gaps of 60, 90 and 120
# days fell past the last transaction and silently produced no window at all,
# so the first run measured decay over 30 days and reported it as the curve.
train_end = dt.quantile(0.45)
GAPS_DAYS = [0, 15, 30, 60, 90, 120]
WINDOW_DAYS = 30
SECONDS = 86_400

In [60]:
key = EntityKey(columns=("card1", "addr1"), anchors=(Anchor("D1"),))
train_raw = full.filter(pl.col("TransactionDT") <= train_end)


def as_split(frame):
    return split_with_contract(
        frame, contract,
        seen_in_train=seen_entity_flag(train_raw, frame, key).fill_null(False).cast(pl.Boolean),
        derivations=rules.derivations,
    )


windows = {}
for gap in GAPS_DAYS:
    lo = train_end + gap * SECONDS
    hi = lo + WINDOW_DAYS * SECONDS
    w = full.filter((pl.col("TransactionDT") > lo) & (pl.col("TransactionDT") <= hi))
    if len(w) >= MIN_ROWS and w["isFraud"].sum() >= MIN_POSITIVES:
        windows[gap] = w
print({g: len(w) for g, w in windows.items()})
missing = [g for g in GAPS_DAYS if g not in windows]
if missing:
    print(f"NO WINDOW for gaps {missing} -- the axis ran out. The curve below\n"
          f"covers {sorted(windows)} only; do not read it as decay out to "
          f"{max(GAPS_DAYS)} days.")

# Fitted once. The validation window the *model* uses for early stopping is the gap-0
# window; every other window is pure holdout, which is what makes the comparison fair.
model = train_lightgbm(as_split(train_raw), as_split(windows[0]), as_split(windows[0]),
                       search_space=None, n_iter=1, seed=42)

decay = []
for gap, w in windows.items():
    split = as_split(w)
    s = model.booster.predict(to_lightgbm(split.features.select(model.booster.feature_name())),
                              num_iteration=model.booster.best_iteration)
    y = split.labels.to_numpy()
    decay.append({"gap_days": gap, "rows": len(w), "base_rate": round(float(y.mean()), 4),
                  "roc_auc": round(float(roc_auc_score(y, s)), 4),
                  "pr_auc": round(float(average_precision_score(y, s)), 4)})
decay = pl.DataFrame(decay)
decay = decay.with_columns(
    roc_drop=(decay["roc_auc"][0] - pl.col("roc_auc")).round(4),
    pr_drop=(decay["pr_auc"][0] - pl.col("pr_auc")).round(4),
)

decay

{0: 89588, 15: 96708, 30: 98027, 60: 55982}
NO WINDOW for gaps [90, 120] -- the axis ran out. The curve below
covers [0, 15, 30, 60] only; do not read it as decay out to 120 days.


gap_days,rows,base_rate,roc_auc,pr_auc,roc_drop,pr_drop
i64,i64,f64,f64,f64,f64,f64
0,89588,0.0417,0.8902,0.5646,0.0,0.0
15,96708,0.0372,0.8603,0.4815,0.0299,0.0831
30,98027,0.0361,0.8461,0.4437,0.0441,0.1209
60,55982,0.0416,0.8618,0.5198,0.0284,0.0448


### The curve measured composition, not time — 2026-08-16

| gap | base rate | ROC-AUC |
| ---: | ---: | ---: |
| 0 | 0.0417 | 0.8902 |
| 15 | 0.0372 | 0.8603 |
| 30 | 0.0361 | 0.8461 |
| 60 | 0.0416 | **0.8618** |

Ordered by **gap**: 0.8902, 0.8603, 0.8461, 0.8618 — not monotone; it recovers at 60 days.
Ordered by **base rate**: 0.8461, 0.8603, 0.8618, 0.8902 — **monotone, four out of four.**

Performance tracks what is *in* the window, not how far the window is from training. The
60-day window has almost exactly the base rate of the 0-day one (0.0416 against 0.0417)
and almost recovers its score.

> **This retracts a claim made from the first run of this notebook** — that shifting the
> validation window by 15 days costs 0.012 ROC-AUC, "four times more than anything we did
> to features". The drop is real; the attribution was not. The reading rule two cells up
> said to check `base_rate` before concluding anything, and it was not applied.

The cell below is the control. Each window is subsampled to a common fraud rate by
dropping negatives at random, so the composition is held fixed and only the distance from
training moves. **If the decay survives this, it is decay. If it disappears, the embargo
was being justified by the wrong number.**


In [ ]:
TARGET_RATE = min(float(w['isFraud'].mean()) for w in windows.values())
rng = np.random.default_rng(0)


def at_common_base_rate(w: pl.DataFrame, rate: float) -> pl.DataFrame:
    """Drop negatives at random until the window hits `rate`.

    Negatives rather than positives: positives are the scarce class and throwing them away
    would trade the confound for a variance problem. Dropping negatives changes the
    prevalence without touching which frauds are in the window, which is the thing being
    held constant.
    """
    pos = w.filter(pl.col('isFraud') == 1)
    neg = w.filter(pl.col('isFraud') == 0)
    keep = min(len(neg), round(len(pos) * (1 - rate) / rate))
    idx = rng.choice(len(neg), size=keep, replace=False)
    return pl.concat([pos, neg[np.sort(idx)]])


controlled = []
for gap, w in windows.items():
    balanced = at_common_base_rate(w, TARGET_RATE)
    split = as_split(balanced)
    s = model.booster.predict(
        to_lightgbm(split.features.select(model.booster.feature_name())),
        num_iteration=model.booster.best_iteration,
    )
    y = split.labels.to_numpy()
    controlled.append({'gap_days': gap, 'rows': len(balanced),
                       'base_rate': round(float(y.mean()), 4),
                       'roc_auc': round(float(roc_auc_score(y, s)), 4),
                       'pr_auc': round(float(average_precision_score(y, s)), 4)})

controlled = pl.DataFrame(controlled)
controlled = controlled.with_columns(
    roc_drop=(controlled['roc_auc'][0] - pl.col('roc_auc')).round(4),
    pr_drop=(controlled['pr_auc'][0] - pl.col('pr_auc')).round(4),
)
print(f'all windows held at a {TARGET_RATE:.4f} fraud rate')
print(controlled)


In [ ]:
# Second control: is the composition shift a *product mix* shift? W is 77% of traffic and
# the products differ enormously in base rate (W 0.0198, C 0.1467), so a few points of mix
# movement between windows moves the pooled rate on its own.
mix = pl.DataFrame([
    {'gap_days': gap, 'rows': len(w),
     **{f'share_{p}': round(float((w['ProductCD'] == p).mean()), 4)
        for p in ['W', 'C', 'R', 'S', 'H']}}
    for gap, w in windows.items()
])
print(mix)

# Third: adversarial validation between the nearest and furthest window. A high AUC means
# a classifier can tell the two windows apart from the features alone -- covariate shift,
# measured rather than assumed. The notebook used to say this separation was impossible on
# this dataset; it is partially possible, and this is how.
from fraud_detection.evaluation.distribution_shift import adversarial_auc

near, far = windows[min(windows)], windows[max(windows)]
shift_auc, importances = adversarial_auc(
    as_split(near).features, as_split(far).features,
    columns=list(model.booster.feature_name()),
)
print(f'\nadversarial AUC, window {min(windows)}d vs {max(windows)}d: {shift_auc:.4f}')
print('0.5 means the windows are indistinguishable; 1.0 means a different population.')
print(importances.head(10))


### Reading the decay curve

| What the curve shows | What it means for the embargo |
| --- | --- |
| `roc_drop` stays inside 0.005 out to 120 days | The current 10% gap is generous, and could be narrowed to buy back training rows |
| `roc_drop` exceeds 0.005 well before the gap width | The gap is doing real work and **validation without it was flattering the model** |
| The curve is not monotone | Base rate is moving between windows, not model quality — check the `base_rate` column before concluding anything |

**The confound to state before drawing conclusions:** later windows are further from
training in calendar time *and* their entities are older. This design cannot separate
"labels went stale" from "the population moved", and no amount of extra windows will —
separating them needs a second axis the dataset does not carry. Record the number as decay,
not as label staleness.

## 4. What this notebook is for

Two outputs, both belonging in `MEASUREMENTS.md`:

1. **The worst segment, named**, with rows, base rate and PR-AUC lift — and one sentence on
   whether a feature is worth building for it.
2. **The decay curve**, with an explicit verdict on whether 10% is too wide, too narrow, or
   the right order of magnitude.

Neither depends on the leaderboard, which is the point: after three consecutive experiments
landing inside the noise band, the useful questions are the ones a single run can still
answer.